# Module 4

## Prerequisites

In [1]:
import boto3
import json
import time
from datetime import datetime
import os

bedrock = boto3.client('bedrock-runtime', region_name='us-east-1')

## Inference profile

In [2]:
prompt = """
What design patterns are used in this code?

You are an expert Python developer. Here's a codebase context:

class DataProcessor:
    def __init__(self, config):
        self.config = config
        self.data = []
    
    def load_data(self, source):
        # Complex data loading logic
        pass
    
    def transform_data(self, transformations):
        # Data transformation pipeline
        pass
    
    def validate_data(self, rules):
        # Data validation logic
        pass

This class is part of a larger system with 50+ similar classes.
Always reference this context when answering questions.
"""

### Invoking cross-region inference profile

In [3]:
response = bedrock.converse(
    modelId='us.amazon.nova-lite-v1:0',
    messages=[
       {
            "role": "user",
            "content": [
                {
                    "text": prompt
                }
          ]
      }
    ]
)

print(json.dumps(response, indent=2))

{
  "ResponseMetadata": {
    "RequestId": "45752412-da7e-4e0c-bb00-12597a4a59da",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "date": "Mon, 02 Mar 2026 13:01:17 GMT",
      "content-type": "application/json",
      "content-length": "4841",
      "connection": "keep-alive",
      "x-amzn-requestid": "45752412-da7e-4e0c-bb00-12597a4a59da"
    },
    "RetryAttempts": 0
  },
  "output": {
    "message": {
      "role": "assistant",
      "content": [
        {
          "text": "Given the provided code snippet and the context that this class is part of a larger system with 50+ similar classes, we can identify a few design patterns that are likely being used or could be implemented to improve the design. Here are some potential design patterns:\n\n### 1. **Factory Pattern**\nSince there are many similar classes, a Factory Pattern can be used to create instances of these classes based on some configuration or input parameters. This would help in managing the instantiation logic i

### Use your own inference profile

In [4]:
# create an inference profile
bedrock_control = boto3.client('bedrock', region_name='us-east-1')

response = bedrock_control.create_inference_profile(
    inferenceProfileName='my-nova-lite-profile',
    description='Custom inference profile for Nova Lite',
    modelSource={
        'copyFrom': 'arn:aws:bedrock:us-east-1:767397992380:inference-profile/us.amazon.nova-lite-v1:0'
    }
)
inference_profile_arn = response['inferenceProfileArn']

print(json.dumps(response, indent=2))

{
  "ResponseMetadata": {
    "RequestId": "0b8ff500-8b9f-4e85-88a0-5dc29303d202",
    "HTTPStatusCode": 201,
    "HTTPHeaders": {
      "date": "Mon, 02 Mar 2026 13:01:25 GMT",
      "content-type": "application/json",
      "content-length": "125",
      "connection": "keep-alive",
      "x-amzn-requestid": "0b8ff500-8b9f-4e85-88a0-5dc29303d202"
    },
    "RetryAttempts": 0
  },
  "inferenceProfileArn": "arn:aws:bedrock:us-east-1:767397992380:application-inference-profile/3wlbgvvvirci",
  "status": "ACTIVE"
}


In [5]:
response = bedrock.converse(
    modelId=inference_profile_arn,
    messages=[
       {
            "role": "user",
            "content": [
                {
                    "text": prompt
                }
          ]
      }
    ]
)

print(json.dumps(response, indent=2))

{
  "ResponseMetadata": {
    "RequestId": "b2501d1b-3b65-4b88-958d-435cfce1513c",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "date": "Mon, 02 Mar 2026 13:01:48 GMT",
      "content-type": "application/json",
      "content-length": "3124",
      "connection": "keep-alive",
      "x-amzn-requestid": "b2501d1b-3b65-4b88-958d-435cfce1513c"
    },
    "RetryAttempts": 0
  },
  "output": {
    "message": {
      "role": "assistant",
      "content": [
        {
          "text": "Based on the provided code snippet, the `DataProcessor` class demonstrates a few design patterns. Here are some of the design patterns that can be identified:\n\n### 1. **Abstract Factory Pattern**\nWhile the provided code does not explicitly demonstrate the Abstract Factory Pattern, it is likely that the larger system with 50+ similar classes might use this pattern to create families of related or dependent objects without specifying their concrete classes.\n\n### 2. **Builder Pattern**\nIf the initial

In [6]:
response = bedrock_control.delete_inference_profile(
    inferenceProfileIdentifier=inference_profile_arn
)

## Batch Inference

Upload jsonl file to S3

In [8]:
bucket_name = "batch-data-inference-dk01"

s3 = boto3.client('s3')

# Upload manifest to S3
input_key = 'input/city-reviews-manifest.jsonl'
s3.upload_file('input/city-reviews-manifest.jsonl', bucket_name, input_key)

Call the CreateModelInvocationJob API

In [9]:
accountnumber = "767397992380"

# Create the bedrock control plane client
bedrock_control = boto3.client('bedrock', region_name='us-east-1')

# Create batch inference job
job_name = f"batch-job-{datetime.now().strftime('%Y%m%d-%H%M%S')}"

response = bedrock_control.create_model_invocation_job(
    jobName=job_name,
    roleArn=f"arn:aws:iam::{accountnumber}:role/BedrockBatchRole",
    modelId="us.amazon.nova-lite-v1:0",
    inputDataConfig={
        's3InputDataConfig': {
            's3Uri': f's3://{bucket_name}/input/'
        }
    },
    outputDataConfig={
        's3OutputDataConfig': {
            's3Uri': f's3://{bucket_name}/output/'
        }
    }
)

job_arn = response['jobArn']

Get the Job Status

In [10]:
status_response = bedrock_control.get_model_invocation_job(jobIdentifier=job_arn)
status = status_response['status']
print(f"Job status: {status}")

Job status: Submitted


Instead of waiting, let's use a jobArn for a job we know already finished

In [11]:
completed_job_arn = job_arn

Wait for the job to finish

In [12]:
# Wait for job completion
while True:
    try:
        status_response = bedrock_control.get_model_invocation_job(
            jobIdentifier=completed_job_arn
        )
    except bedrock_control.exceptions.ResourceNotFoundException:
        print("Job not found yet, retrying in 10 seconds...")
        time.sleep(10)
        continue

    status = status_response['status']
    print(f"Job status: {status}")

    if status in ['Completed', 'PartiallyCompleted', 'Failed', 'Stopped', 'Expired']:
        break

    time.sleep(30)



Job status: Failed


Download the result

In [13]:
if status in ["Completed", "PartiallyCompleted"]:
    # Download results
    output_objects = s3.list_objects_v2(Bucket=bucket_name, Prefix='output/')
    
    os.makedirs('batch-output', exist_ok=True)
    
    for obj in output_objects.get('Contents', []):
        if obj['Key'].endswith('/'):
            continue
            
        local_file = os.path.join('batch-output', obj['Key'].split('/')[-1])
        s3.download_file(bucket_name, obj['Key'], local_file)
        print(f"Downloaded: {local_file}")


Stop the job we created to save on cost

In [ ]:
response = bedrock_control.stop_model_invocation_job(
    jobIdentifier=job_arn
)

## Prompt Caching with Nova Lite

In [22]:
# Large context that we want to cache
cached_context = """
You are a expert Python developer. Here's a large codebase context:

class DataProcessor:
    def __init__(self, config):
        self.config = config
        self.data = []
    
    def load_data(self, source):
        # Complex data loading logic
        pass
    
    def transform_data(self, transformations):
        # Data transformation pipeline
        pass
    
    def validate_data(self, rules):
        # Data validation logic
        pass

This class is part of a larger system with 50+ similar classes.
Always reference this context when answering questions.
"""

First request that establishes the cache

In [23]:
firstRequest = bedrock.converse(
    modelId='amazon.nova-lite-v1:0',
    messages=[
       {
            "role": "user",
            "content": [
                {
                    "text": cached_context
                },
                {
                    "cachePoint": {
                        "type": "default"
                    }
                },
                {
                    "text": "What design patterns are used in this code?"
                }
          ]
      }
    ]
)

print(json.dumps(firstRequest, indent=2))

{
  "ResponseMetadata": {
    "RequestId": "ad80f3e2-6f96-4b09-b37f-ebd43737ce19",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "date": "Fri, 12 Dec 2025 12:08:07 GMT",
      "content-type": "application/json",
      "content-length": "3794",
      "connection": "keep-alive",
      "x-amzn-requestid": "ad80f3e2-6f96-4b09-b37f-ebd43737ce19"
    },
    "RetryAttempts": 0
  },
  "output": {
    "message": {
      "role": "assistant",
      "content": [
        {
          "text": "Given the provided context, the `DataProcessor` class exhibits several design patterns. Here's a breakdown of the design patterns used:\n\n### 1. **Initialization and Configuration (Builder Pattern)**\nThe `DataProcessor` class uses an initialization method (`__init__`) that takes a `config` parameter. This pattern is often associated with the **Builder Pattern**, where the initialization of an object is separated from its construction. This allows for more complex configurations and initializations.\n\

Second request that uses cached context

In [24]:
secondRequest = bedrock.converse(
    modelId='amazon.nova-lite-v1:0',
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "text": cached_context
                },
                {
                    "cachePoint": {
                        "type": "default"
                    }
                },
                {
                    "text": "what else can you tell me about this code?"
                }
          ]
      }
    ]
)

print(json.dumps(secondRequest, indent=2))

{
  "ResponseMetadata": {
    "RequestId": "ea15ef95-7ee4-4872-b7d3-6732e510579b",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "date": "Fri, 12 Dec 2025 12:08:21 GMT",
      "content-type": "application/json",
      "content-length": "8011",
      "connection": "keep-alive",
      "x-amzn-requestid": "ea15ef95-7ee4-4872-b7d3-6732e510579b"
    },
    "RetryAttempts": 0
  },
  "output": {
    "message": {
      "role": "assistant",
      "content": [
        {
          "text": "Certainly! Let's break down the provided code and discuss some potential enhancements, considerations, and best practices based on the context of a large codebase with many similar classes.\n\n### Key Points\n\n1. **Class Initialization**:\n    ```python\n    def __init__(self, config):\n        self.config = config\n        self.data = []\n    ```\n    - **Initialization**: The `DataProcessor` class is initialized with a `config` parameter, which is expected to be a dictionary or a configuration object

Compare the usage

In [25]:
print("### First request ###")
print("Usage:", json.dumps(firstRequest['usage'], indent=2))
print("Metrics:", json.dumps(firstRequest['metrics'], indent=2))

print("\n\n### Second request ###")
print("Usage:", json.dumps(secondRequest['usage'], indent=2))
print("Metrics:", json.dumps(secondRequest['metrics'], indent=2))

### First request ###
Usage: {
  "inputTokens": 9,
  "outputTokens": 702,
  "totalTokens": 835,
  "cacheReadInputTokens": 0,
  "cacheWriteInputTokens": 124
}
Metrics: {
  "latencyMs": 5171
}


### Second request ###
Usage: {
  "inputTokens": 10,
  "outputTokens": 1665,
  "totalTokens": 1799,
  "cacheReadInputTokens": 124,
  "cacheWriteInputTokens": 0
}
Metrics: {
  "latencyMs": 9566
}


## Converse on documents
### Using `bytes` as a source

In [26]:
with open('input/AnyCompany_financial_10K.pdf', "rb") as file:
    doc_bytes = file.read()

In [27]:
response = bedrock.converse(
    modelId='us.amazon.nova-lite-v1:0',
    messages=[
       {
            "role": "user",
            "content": [
                {
                    "document": {
                        "format": "pdf",
                        "name": "AnyCompany_financial",
                        "source": {
                            "bytes": doc_bytes
                        }
                    }
                },
                {
                    "text": "What investments have AnyCompany made?"
                }
          ]
      }
    ]
)

print(json.dumps(response, indent=2))

{
  "ResponseMetadata": {
    "RequestId": "87a8d3c7-7547-4cc2-8846-7a8090a58704",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "date": "Fri, 12 Dec 2025 12:10:54 GMT",
      "content-type": "application/json",
      "content-length": "242",
      "connection": "keep-alive",
      "x-amzn-requestid": "87a8d3c7-7547-4cc2-8846-7a8090a58704"
    },
    "RetryAttempts": 0
  },
  "output": {
    "message": {
      "role": "assistant",
      "content": [
        {
          "text": "Company A, Company B, Company C"
        }
      ]
    }
  },
  "stopReason": "end_turn",
  "usage": {
    "inputTokens": 194688,
    "outputTokens": 8,
    "totalTokens": 194696
  },
  "metrics": {
    "latencyMs": 47246
  }
}


### Using s3Location as a source

In [28]:
bucket_name = "batch-data-inference-dk01"

s3 = boto3.client('s3')

input_key = 'AnyCompany_financial_10K.pdf'
s3.upload_file('input/AnyCompany_financial_10K.pdf', bucket_name, input_key)

In [29]:
response = bedrock.converse(
    modelId='us.amazon.nova-lite-v1:0',
    messages=[
       {
            "role": "user",
            "content": [
                {
                    "document": {
                        "format": "pdf",
                        "name": "AnyCompany_financial",
                        "source": {
                            "s3Location": {
                                "uri": f"s3://{bucket_name}/{input_key}"
                            }
                        }
                    }
                },
                {
                    "text": "What property does AnyCompany own?"
                }
          ]
      }
    ]
)

print(json.dumps(response, indent=2))

{
  "ResponseMetadata": {
    "RequestId": "2258f474-1b19-46af-877a-37842d714a49",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "date": "Fri, 12 Dec 2025 12:15:27 GMT",
      "content-type": "application/json",
      "content-length": "340",
      "connection": "keep-alive",
      "x-amzn-requestid": "2258f474-1b19-46af-877a-37842d714a49"
    },
    "RetryAttempts": 0
  },
  "output": {
    "message": {
      "role": "assistant",
      "content": [
        {
          "text": "AnyCompany owns a diverse portfolio of properties, including headquarters, regional offices, data centers, and other facilities."
        }
      ]
    }
  },
  "stopReason": "end_turn",
  "usage": {
    "inputTokens": 194688,
    "outputTokens": 22,
    "totalTokens": 194710
  },
  "metrics": {
    "latencyMs": 49054
  }
}


## Conversation history

### Create the DynamoDB Table and set the time to live

In [44]:
table_name = "conversation-history"

dynamodb = boto3.client('dynamodb', region_name="us-east-1")
response = dynamodb.create_table(
    TableName=table_name,
    BillingMode='PAY_PER_REQUEST',
    AttributeDefinitions=[
        {
            'AttributeName': 'userId',
            'AttributeType': 'S'
        },
        {
            'AttributeName': 'timestamp',
            'AttributeType': 'N'
        }
    ],
    KeySchema=[
        {
            'AttributeName': 'userId',
            'KeyType': 'HASH'
        },
        {
            'AttributeName': 'timestamp',
            'KeyType': 'RANGE'
        }
    ]
)

# Wait for table to be active
waiter = dynamodb.get_waiter('table_exists')
waiter.wait(TableName=table_name)

ttl_response = dynamodb.update_time_to_live(
    TableName=table_name,
    TimeToLiveSpecification={
        'AttributeName': 'ttl',
        'Enabled': True
    }
)

### Create the functions to get the conversation history and store new messages

In [45]:
table_name = "conversation-history"
ddbclient = boto3.client('dynamodb', region_name = "us-east-1")

def get_conversation_history(user_id):
    # Pagination is required in case the conversation is more than 1MB.
    paginator = ddbclient.get_paginator('query')
    pages = paginator.paginate(
        TableName = table_name,
        KeyConditionExpression = 'userId = :val',
        ExpressionAttributeValues = {':val': {'S': user_id}}
    )
    messages = []
    for page in pages:
        for item in page.get('Items', []):
            messages.append({
                "role": item["role"]["S"],
                "content": [{ "text": item["message"]["S"] }]
            })
    return messages

In [46]:
dynamodbResource = boto3.resource('dynamodb', region_name = "us-east-1")
conversation_table = dynamodbResource.Table(table_name)

def store_message(user_id, message, role):
    now_in_seconds = int(time.time())
    expire_ttl = now_in_seconds + (1 * 24 * 60 * 60) # 1 day
    
    conversation_table.put_item(
        Item = {
            'userId': user_id,
            'timestamp': now_in_seconds,
            'message': message,
            'role': role,
            'ttl': expire_ttl
        }
    )


### Create the function to send messages to the LLM

In [47]:
def send_message(user_id, message):
    
    # Load the conversation history
    conversation_messages = get_conversation_history(user_id)

    # Store the new message
    store_message(user_id, message, "user")
    
    # Add the new message to the conversation
    conversation_messages.append({
        "role": "user",
        "content": [{ "text": message }]
    })

    # Send the conversation to the LLM
    model_response = bedrock.converse(
        modelId="amazon.nova-lite-v1:0",
        messages=conversation_messages,
        system = [{ 
            "text": "Please provide a helpful, conversational response based on the available information and conversation history." 
        }],
        inferenceConfig = {
            "maxTokens": 300,
            "temperature": 0.7,
            "topP": 0.9
        }
    )

    # Parse the message from the LLM, store it and return it
    assistant_msg = model_response['output']['message']['content'][0]['text']
    store_message(user_id, assistant_msg, "assistant")

    return conversation_messages, assistant_msg

### Sending messages

In [48]:
user_id = "john"
convo, response = send_message(user_id, "My name is John and I am a developer. How can I improve my coding skills?")
print(response)


Hey John! It's great to hear that you're looking to improve your coding skills. Here are some actionable steps you can take to level up:

### **1. Practice Regularly**
- **LeetCode/HackerRank:** These platforms offer a variety of problems that can help you practice different coding concepts.
- **Project Euler:** For those interested in math-related problems.
- **Codewars:** Offers coding challenges and allows you to see how others solve problems.

### **2. Learn New Languages and Frameworks**
- **Languages:** Try learning languages like Python, JavaScript, or Go if you haven't already. Each language has its own strengths.
- **Frameworks:** Familiarize yourself with popular frameworks and libraries relevant to your field (e.g., React, Angular for web; Django, Flask for Python).

### **3. Contribute to Open Source**
- **GitHub:** Look for beginner-friendly issues in open-source projects. Contributing can give you real-world experience.
- **Code Review:** Review others' code and get feedb

In [49]:
convo, response = send_message(user_id, "What I do?")
print(response)

It sounds like you're asking for some guidance on what steps you can take to improve your coding skills. Given your background as a developer, here are some tailored suggestions:

### **1. Set Specific Goals**
- **Skill Focus:** Identify specific areas you want to improve in (e.g., algorithms, web development, database management).
- **Project-Based Learning:** Set a goal to complete a project in a new language or framework.

### **2. Deepen Your Understanding**
- **Algorithms and Data Structures:** Strengthen your grasp on fundamental concepts. Books like "Introduction to Algorithms" by Cormen et al. can be helpful.
- **Design Patterns:** Learn common design patterns and when to use them. "Design Patterns: Elements of Reusable Object-Oriented Software" by the Gang of Four is a classic.

### **3. Engage with the Community**
- **Online Forums:** Participate in discussions on Stack Overflow, Reddit (e.g., r/learnprogramming, r/webdev), or specialized forums.
- **Meetups and Conferences:*

In [50]:
print(json.dumps(convo, indent=2))

[
  {
    "role": "user",
    "content": [
      {
        "text": "My name is John and I am a developer. How can I improve my coding skills?"
      }
    ]
  },
  {
    "role": "assistant",
    "content": [
      {
        "text": "Hey John! It's great to hear that you're looking to improve your coding skills. Here are some actionable steps you can take to level up:\n\n### **1. Practice Regularly**\n- **LeetCode/HackerRank:** These platforms offer a variety of problems that can help you practice different coding concepts.\n- **Project Euler:** For those interested in math-related problems.\n- **Codewars:** Offers coding challenges and allows you to see how others solve problems.\n\n### **2. Learn New Languages and Frameworks**\n- **Languages:** Try learning languages like Python, JavaScript, or Go if you haven't already. Each language has its own strengths.\n- **Frameworks:** Familiarize yourself with popular frameworks and libraries relevant to your field (e.g., React, Angular for we